# RAG Pipeline

**RAG (Retrieval-Augmented Generation)** combines a retrieval step with a generation step:

1. **Retrieve** relevant documents from a knowledge base using semantic search
2. **Augment** the prompt with the retrieved context
3. **Generate** an answer conditioned on the context

This notebook uses the reusable modules in `src/` and the project knowledge base in `datasets/`.

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers faiss-cpu pillow -q

## Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

from sentence_transformers import SentenceTransformer
from transformers import pipeline
from vector_search import VectorSearch
import numpy as np
import torch

device = 0 if torch.cuda.is_available() else -1
print(f"PyTorch {torch.__version__} | device: {'GPU' if device == 0 else 'CPU'}")

## 1. Build the Knowledge Base

In [ ]:
corpus_path = os.path.join(os.getcwd(), "..", "datasets", "example_documents.txt")
with open(corpus_path) as f:
    base_docs = [line.strip() for line in f if line.strip()]

extra_docs = [
    "Natural language processing (NLP) is a branch of AI focused on language understanding.",
    "Machine learning models learn patterns from large amounts of training data.",
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "FAISS is a library for efficient similarity search over dense vectors.",
    "RAG combines retrieval from a knowledge base with text generation for accurate answers.",
    "Fine-tuning adjusts pre-trained model weights on task-specific data.",
    "Embeddings are dense vector representations that capture semantic meaning.",
    "GPT-2 is an autoregressive language model that generates text token by token.",
]

documents = base_docs + extra_docs
print(f"Knowledge base: {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"  [{i:2d}] {doc}")

## 2. Embed and Index the Knowledge Base

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(documents, show_progress_bar=True).astype("float32")

vs = VectorSearch(doc_embeddings.shape[1])
vs.add(doc_embeddings)
print(f"\nFAISS index ready: {vs.index.ntotal} vectors ({doc_embeddings.shape[1]} dims)")

## 3. Build the RAG Pipeline

**Retrieve** → **Augment** → **Generate**

In [ ]:
generator = pipeline("text-generation", model="gpt2", device=device)

def rag_answer(question, top_k=3, max_new_tokens=60):
    # Step 1: Retrieve
    q_emb = embedder.encode([question]).astype("float32")
    _, indices = vs.search(q_emb[0], k=top_k)
    retrieved = [documents[i] for i in indices[0]]

    # Step 2: Augment — build context-aware prompt
    context = " ".join(retrieved)
    prompt = f"Context: {context}\n\nQuestion: {question}\nAnswer:"

    # Step 3: Generate
    output = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False,
                       pad_token_id=generator.tokenizer.eos_token_id)
    answer = output[0]["generated_text"][len(prompt):].strip()
    return retrieved, answer

# Test questions
questions = [
    "What is machine learning?",
    "How do transformers process sequences?",
    "What is the purpose of embeddings?",
]

for q in questions:
    retrieved, answer = rag_answer(q)
    print(f"Q: {q}")
    print(f"Retrieved docs: {retrieved[0][:60]}...")
    print(f"A: {answer[:120]}")
    print()

## 4. RAG vs No-Context Generation

Observe how providing context changes the quality of the generated answer.

In [ ]:
question = "What is the role of attention in transformers?"

print("=== WITHOUT context (plain GPT-2) ===")
no_ctx_prompt = f"Question: {question}\nAnswer:"
out = generator(no_ctx_prompt, max_new_tokens=50, do_sample=False,
                pad_token_id=generator.tokenizer.eos_token_id)
print(out[0]["generated_text"][len(no_ctx_prompt):].strip()[:200])

print("\n=== WITH RAG context ===")
_, rag_ans = rag_answer(question)
print(rag_ans[:200])